In [2]:
import os

from nwtrace import *
import pandas as pd
import geopandas as gpd
import numpy as np

from pathlib import Path

In [3]:
york_sewers = Path('data/more/york_region_storm.gpkg')

segments = gpd.read_file(york_sewers, layer="segments")
nodes = gpd.read_file(york_sewers, layer="nodes")

nodes = nodes[["FACILITYID", "AVG_ELEV", "geometry"]]
segments = segments[["FACILITYID", "geometry"]]

In [4]:
network = utils.network_from_geometry(
    segments=segments,
    nodes=nodes,
    segment_id_field="FACILITYID",
    node_id_field="FACILITYID",
    distance_threshold=0.6
)

network

role,FACILITYID,from,to,from_dist,to_dist
0,STMSL0115_0001,CB0115_0002,STMMH0115_0003,0.0,0.0
1,STMSL0115_0002,CB0115_0003,STMMH0115_0003,0.0,0.0
2,STMSL0115_0003,CB10150,STMMH0115_0001,0.0,0.0
3,STMSL0115_0004,CB0115_0004,NaN,0.0,NaN
4,STMSL0115_0005,STMMH0115_0001,STMMH12094,0.0,0.0
...,...,...,...,...,...
47276,STMSS99950,STMMH13627,NaN,0.0,NaN
47277,STMSS99956,STMSJ13034,NaN,0.0,NaN
47278,STMSS99957,STMSJ13303,NaN,0.0,NaN
47279,STMSS99958,STMSJ13304,NaN,0.0,NaN


In [8]:
repaired_network = utils.verify_flow_directionality(
    segments=network,
    nodes=nodes,
    segment_id_field="FACILITYID",
    elevation_field="AVG_ELEV",
    upstream_field="from",
    downstream_field="to",
    repair_errors=True
)

repaired_network

role,FACILITYID,from,to,from_dist,to_dist,from_height,to_height
0,STMSL0115_0001,CB0115_0002,STMMH0115_0003,0.0,0.0,173.020,NaN
1,STMSL0115_0002,CB0115_0003,STMMH0115_0003,0.0,0.0,173.290,NaN
2,STMSL0115_0003,CB10150,STMMH0115_0001,0.0,0.0,176.000,NaN
3,STMSL0115_0004,CB0115_0004,NaN,0.0,NaN,NaN,NaN
4,STMSL0115_0005,STMMH0115_0001,STMMH12094,0.0,0.0,NaN,171.5825
...,...,...,...,...,...,...,...
47276,STMSS99950,STMMH13627,NaN,0.0,NaN,238.476,NaN
47277,STMSS99956,STMSJ13034,NaN,0.0,NaN,NaN,NaN
47278,STMSS99957,STMSJ13303,NaN,0.0,NaN,NaN,NaN
47279,STMSS99958,STMSJ13304,NaN,0.0,NaN,NaN,NaN


In [10]:
network.set_index("FACILITYID").loc["STMSL10057"]

role
from           STMMH5444
to             STMMH5443
from_dist            0.0
to_dist              0.0
from_height      176.066
to_height       177.5815
Name: STMSL10057, dtype: object

In [12]:
repaired_network.set_index("FACILITYID").loc["STMSL10057"]

role
from           STMMH5443
to             STMMH5444
from_dist            0.0
to_dist              0.0
from_height     177.5815
to_height        176.066
Name: STMSL10057, dtype: object